# 💰 portfolio — Indian Stock Tracker
Track live NSE stock prices, manage your portfolio, and see your gains/losses.

**Run each cell one by one by pressing the ▶️ button on the left.**

In [ ]:
# ✅ STEP 1 — Install dependencies (run this first!)
!pip install yfinance rich -q

In [ ]:
# ✅ STEP 2 — Load the portfolio tool
import json, os
from datetime import datetime
import yfinance as yf
from rich.console import Console
from rich.table import Table
from rich import box
from rich.panel import Panel
from rich.text import Text

console = Console()
portfolio = {}  # stores your stocks in memory

def to_nse(symbol):
    symbol = symbol.upper().strip()
    if not symbol.endswith('.NS') and not symbol.endswith('.BO'):
        symbol += '.NS'
    return symbol

def price(symbol):
    """📈 Check live price of any NSE stock"""
    nse_symbol = to_nse(symbol)
    console.print(f"\n🔍 Fetching price for [bold cyan]{symbol.upper()}[/bold cyan]...")
    try:
        ticker = yf.Ticker(nse_symbol)
        info = ticker.info
        p = info.get('currentPrice') or info.get('regularMarketPrice')
        name = info.get('longName') or info.get('shortName') or symbol.upper()
        change = info.get('regularMarketChange', 0)
        change_pct = info.get('regularMarketChangePercent', 0)
        day_high = info.get('dayHigh', 'N/A')
        day_low = info.get('dayLow', 'N/A')
        if p is None:
            console.print(f'[red]❌ Could not find price for {symbol}. Check the symbol.[/red]')
            return
        color = 'green' if change >= 0 else 'red'
        arrow = '▲' if change >= 0 else '▼'
        t = Text()
        t.append(f'{name}\n', style='bold white')
        t.append(f'₹{p:,.2f}  ', style=f'bold {color}')
        t.append(f'{arrow} {abs(change):.2f} ({abs(change_pct):.2f}%)\n', style=color)
        t.append(f'\nDay High: ₹{day_high}   Day Low: ₹{day_low}', style='dim')
        console.print(Panel(t, title=f'[bold]{nse_symbol}[/bold]', border_style=color))
    except Exception as e:
        console.print(f'[red]❌ Error: {e}[/red]')

def add(symbol, quantity, buy_price):
    """📁 Add a stock to your portfolio"""
    nse_symbol = to_nse(symbol)
    portfolio[nse_symbol] = {
        'symbol': nse_symbol,
        'quantity': float(quantity),
        'buy_price': float(buy_price),
        'added_on': datetime.now().strftime('%Y-%m-%d')
    }
    invested = float(quantity) * float(buy_price)
    console.print(f'\n✅ Added [bold cyan]{nse_symbol}[/bold cyan] to portfolio.')
    console.print(f'   {quantity} shares @ ₹{float(buy_price):,.2f} = [bold]₹{invested:,.2f}[/bold] invested')

def show():
    """💹 Show portfolio with live gain/loss"""
    if not portfolio:
        console.print('\n[yellow]📭 Portfolio is empty. Use add("RELIANCE", 10, 2500) to add stocks.[/yellow]')
        return
    console.print('\n📊 [bold]Fetching live prices...[/bold]')
    table = Table(title='💰 My Portfolio', box=box.ROUNDED, header_style='bold magenta', border_style='dim')
    table.add_column('Stock', style='bold cyan', no_wrap=True)
    table.add_column('Qty', justify='right')
    table.add_column('Buy Price', justify='right')
    table.add_column('Current', justify='right')
    table.add_column('Invested', justify='right')
    table.add_column('Current Value', justify='right')
    table.add_column('Gain / Loss', justify='right')
    table.add_column('Return %', justify='right')
    total_invested = total_current = 0
    for symbol, entry in portfolio.items():
        qty = entry['quantity']
        buy_price = entry['buy_price']
        invested = qty * buy_price
        try:
            info = yf.Ticker(symbol).info
            current_price = info.get('currentPrice') or info.get('regularMarketPrice') or buy_price
        except:
            current_price = buy_price
        current_value = qty * current_price
        gain = current_value - invested
        gain_pct = (gain / invested) * 100
        total_invested += invested
        total_current += current_value
        color = 'green' if gain >= 0 else 'red'
        table.add_row(
            symbol.replace('.NS', ''),
            str(int(qty)),
            f'₹{buy_price:,.2f}',
            f'₹{current_price:,.2f}',
            f'₹{invested:,.2f}',
            f'₹{current_value:,.2f}',
            f'[{color}]₹{gain:+,.2f}[/{color}]',
            f'[{color}]{gain_pct:+.2f}%[/{color}]'
        )
    console.print(table)
    total_gain = total_current - total_invested
    total_pct = (total_gain / total_invested) * 100 if total_invested else 0
    color = 'green' if total_gain >= 0 else 'red'
    arrow = '▲' if total_gain >= 0 else '▼'
    console.print(Panel(
        f'Total Invested: [bold]₹{total_invested:,.2f}[/bold]   '
        f'Current Value: [bold]₹{total_current:,.2f}[/bold]   '
        f'Overall P&L: [{color}]{arrow} ₹{abs(total_gain):,.2f} ({abs(total_pct):.2f}%)[/{color}]',
        title='[bold]Portfolio Summary[/bold]', border_style=color
    ))

def remove(symbol):
    """🗑️ Remove a stock from portfolio"""
    nse_symbol = to_nse(symbol)
    if nse_symbol in portfolio:
        del portfolio[nse_symbol]
        console.print(f'\n🗑️ Removed [bold cyan]{nse_symbol}[/bold cyan] from portfolio.')
    else:
        console.print(f'\n[yellow]{nse_symbol} not found in portfolio.[/yellow]')

console.print(Panel('[bold green]✅ Portfolio tool loaded! Scroll down to use it.[/bold green]', border_style='green'))

In [ ]:
# 📈 FEATURE 1 — Check live price
# Change the symbol to any NSE stock you want!
price("RELIANCE")

In [ ]:
# 📁 FEATURE 2 — Add stocks to your portfolio
# Format: add("SYMBOL", quantity, buy_price)
add("RELIANCE", 10, 2500)
add("TCS", 5, 3800)
add("INFY", 8, 1500)

In [ ]:
# 💹 FEATURE 3 — View portfolio with live gain/loss
show()

In [ ]:
# 🗑️ BONUS — Remove a stock
remove("INFY")